# Fake News Detection Using Artificial Intelligence
## Complete Enhanced Pipeline — Kaggle Version

**MSc Project | Arafat Hazrati | Student ID: 24015414**  
**London Metropolitan University | 2024–2025**  
**Supervisor: Professor Karim Ouazzane**

---

### Features
| Feature | Description |
|---------|-------------|
| ✅ Confidence Scores | Probability scores for every prediction |
| ✅ Uncertainty Handling | Four-tier decision system |
| ✅ Real-World Evaluation | Cross-dataset testing |
| ✅ Error Analysis | Top false positives and negatives |
| ✅ LIME Explainability | Which words drove each prediction |
| ✅ All 6 Models | NB, LR, SVM, RF, BiLSTM, RoBERTa |

### Instructions
1. GPU T4 x2 must be selected in Session options
2. Dataset fake-news-isot must be attached
3. Click Save Version → Save and Run All (Commit)
4. Come back in ~2 hours — everything saves to Output tab

---
## Section 1: Install Packages

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn nltk \
             tensorflow torch transformers datasets \
             wordcloud imbalanced-learn lime -q
print('All packages installed successfully')

---
## Section 2: Imports and Configuration

In [ ]:
import os, re, time, pickle, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, LSTM, Bidirectional, Dense,
    Dropout, GlobalMaxPooling1D, Conv1D, Input)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
from lime.lime_text import LimeTextExplainer

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

OUTPUT = '/kaggle/working/'

# Configuration
CONFIG = {
    'test_size'            : 0.20,
    'val_size'             : 0.10,
    'random_state'         : SEED,
    'tfidf_max_features'   : 50000,
    'tfidf_ngram_range'    : (1, 2),
    'max_sequence_length'  : 512,
    'vocab_size'           : 20000,
    'embedding_dim'        : 128,
    'lstm_units'           : 128,
    'lstm_dropout'         : 0.3,
    'batch_size'           : 32,
    'lstm_epochs'          : 10,
    'roberta_model'        : 'distilroberta-base',
    'roberta_epochs'       : 3,
    'roberta_batch_size'   : 16,
    'roberta_lr'           : 2e-5,
    'cv_folds'             : 5,
    'confidence_high'      : 0.85,
    'confidence_medium'    : 0.65,
    'confidence_uncertain' : 0.55,
    'lime_num_features'    : 12,
    'lime_num_samples'     : 300,
    'error_analysis_n'     : 10,
}

COLORS = {
    'real': '#2E75B6', 'fake': '#C00000',
    'uncertain': '#FFC000', 'dark': '#1F3864',
    'green': '#70AD47', 'purple': '#7030A0', 'grey': '#888888'
}

print('All libraries imported')
print(f'TensorFlow GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'PyTorch CUDA  : {torch.cuda.is_available()}')
print('Pipeline ready')

---
## Section 3: Data Loading

In [ ]:
print('[STEP 1] Loading Dataset...')

# Kaggle dataset path — arrouu/fake-news-isot
FAKE_PATH = '/kaggle/input/datasets/arrouu/fake-news-isot/Fake.csv'
TRUE_PATH = '/kaggle/input/datasets/arrouu/fake-news-isot/True.csv'

# Auto-detect if paths differ
if not os.path.exists(FAKE_PATH):
    print('Searching for CSV files...')
    for root, dirs, files_list in os.walk('/kaggle/input'):
        for f in files_list:
            full = os.path.join(root, f)
            if f.lower() == 'fake.csv': FAKE_PATH = full
            if f.lower() == 'true.csv': TRUE_PATH = full

print(f'Fake.csv: {FAKE_PATH}')
print(f'True.csv: {TRUE_PATH}')

fake_df = pd.read_csv(FAKE_PATH)
true_df = pd.read_csv(TRUE_PATH)

fake_df['label'] = 1
true_df['label'] = 0

df = pd.concat([fake_df, true_df], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Total   : {len(df):,}')
print(f'Real    : {len(df[df["label"]==0]):,}')
print(f'Fake    : {len(df[df["label"]==1]):,}')
print('Dataset loaded')

---
## Section 4: Exploratory Data Analysis

In [ ]:
print('[STEP 2] Exploratory Data Analysis...')

df['text_length']  = df['text'].astype(str).apply(lambda x: len(x.split()))
df['title_length'] = df['title'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Exploratory Data Analysis — ISOT Fake News Dataset',
             fontsize=15, fontweight='bold')

# Class distribution
ax1 = axes[0,0]
counts = df['label'].value_counts()
ax1.pie(counts, labels=['Real','Fake'], autopct='%1.1f%%',
        colors=[COLORS['real'], COLORS['fake']],
        startangle=90, explode=(0.05,0.05))
ax1.set_title('Class Distribution', fontweight='bold')

# Article length
ax2 = axes[0,1]
real_l = df[df['label']==0]['text_length']
fake_l = df[df['label']==1]['text_length']
ax2.hist(real_l.clip(0,2000), bins=50, alpha=0.7,
         color=COLORS['real'], label='Real', density=True)
ax2.hist(fake_l.clip(0,2000), bins=50, alpha=0.7,
         color=COLORS['fake'], label='Fake', density=True)
ax2.set_title('Article Length Distribution', fontweight='bold')
ax2.legend()

# Subject
ax3 = axes[1,0]
if 'subject' in df.columns:
    sub = df.groupby(['subject','label']).size().unstack(fill_value=0)
    sub.plot(kind='bar', ax=ax3,
             color=[COLORS['real'],COLORS['fake']], alpha=0.85)
    ax3.set_title('Articles by Subject', fontweight='bold')
    ax3.legend(['Real','Fake'])
    ax3.tick_params(axis='x', rotation=45)
else:
    ax3.text(0.5,0.5,'Subject column not available',
             ha='center',va='center',transform=ax3.transAxes)

# Title length
ax4 = axes[1,1]
bp = ax4.boxplot(
    [df[df['label']==0]['title_length'].clip(0,50),
     df[df['label']==1]['title_length'].clip(0,50)],
    labels=['Real','Fake'], patch_artist=True, notch=True)
bp['boxes'][0].set_facecolor(COLORS['real'])
bp['boxes'][1].set_facecolor(COLORS['fake'])
ax4.set_title('Title Length Comparison', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT+'01_eda.png', dpi=150, bbox_inches='tight')
plt.close()
print('EDA complete — chart saved')

---
## Section 5: Text Preprocessing

In [ ]:
print('[STEP 3] Text Preprocessing...')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\(reuters\)|\(ap\)|\(afp\)|\(cnn\)', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [lemmatizer.lemmatize(w)
             for w in words
             if w not in stop_words and len(w) > 2]
    return ' '.join(words)

df['raw_content'] = df['title'].astype(str) + ' ' + df['text'].astype(str)

print('Cleaning text — approximately 2-3 minutes...')
start = time.time()
df['clean_content'] = df['raw_content'].apply(clean_text)
elapsed = time.time() - start

before = len(df)
df = df[df['clean_content'].str.split().str.len() >= 10].reset_index(drop=True)
print(f'Done in {elapsed:.1f}s')
print(f'Articles removed (too short): {before - len(df)}')
print(f'Articles remaining: {len(df):,}')

# Word clouds
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
real_txt = ' '.join(df[df['label']==0]['clean_content'])
fake_txt = ' '.join(df[df['label']==1]['clean_content'])
wc_r = WordCloud(width=900, height=600, background_color='white',
                 colormap='Blues', max_words=150).generate(real_txt)
wc_f = WordCloud(width=900, height=600, background_color='white',
                 colormap='Reds', max_words=150).generate(fake_txt)
ax1.imshow(wc_r, interpolation='bilinear'); ax1.axis('off')
ax1.set_title('Most Common Words — Real News', fontweight='bold', fontsize=13)
ax2.imshow(wc_f, interpolation='bilinear'); ax2.axis('off')
ax2.set_title('Most Common Words — Fake News', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT+'02_wordclouds.png', dpi=150, bbox_inches='tight')
plt.close()
print('Preprocessing complete')

---
## Section 6: Train-Test Split and TF-IDF

In [ ]:
print('[STEP 4] Splitting and Feature Engineering...')

X = df['clean_content']
y = df['label']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=CONFIG['test_size'],
    random_state=SEED, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=CONFIG['val_size']/(1-CONFIG['test_size']),
    random_state=SEED, stratify=y_temp)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

tfidf = TfidfVectorizer(
    max_features=CONFIG['tfidf_max_features'],
    ngram_range=CONFIG['tfidf_ngram_range'],
    min_df=2, max_df=0.95,
    sublinear_tf=True, analyzer='word',
    token_pattern=r'\b[a-zA-Z]{2,}\b')

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Vocabulary: {len(tfidf.vocabulary_):,} features')
print('Split and TF-IDF complete')

---
## Section 7: Confidence and Uncertainty Framework

In [ ]:
print('[STEP 5] Setting Up Confidence Framework...')

def get_confidence_tier(confidence_fake):
    conf_real = 1 - confidence_fake
    max_conf  = max(confidence_fake, conf_real)
    if max_conf < CONFIG['confidence_uncertain']:
        return {'tier':'UNCERTAIN','label':'Cannot Determine',
                'action':'Verify with multiple sources','emoji':'?'}
    elif max_conf < CONFIG['confidence_medium']:
        verdict = 'Likely Fake' if confidence_fake > 0.5 else 'Likely Real'
        return {'tier':'LOW_CONFIDENCE','label':verdict,
                'action':'Treat with caution','emoji':'!'}
    elif max_conf < CONFIG['confidence_high']:
        verdict = 'Probably Fake' if confidence_fake > 0.5 else 'Probably Real'
        return {'tier':'MEDIUM_CONFIDENCE','label':verdict,
                'action':'Cross-check recommended','emoji':'~'}
    else:
        verdict = 'FAKE NEWS' if confidence_fake > 0.5 else 'REAL NEWS'
        return {'tier':'HIGH_CONFIDENCE','label':verdict,
                'action':'High confidence verdict','emoji':'!'}

all_results = []
all_probs   = {}
all_preds   = {}

def full_evaluation(name, model, X_tr, y_tr, X_te, y_te,
                    cv_X=None, cv_y=None):
    start  = time.time()
    y_pred = model.predict(X_te)
    elapsed = time.time() - start

    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        y_prob = y_pred.astype(float)

    all_probs[name] = y_prob
    all_preds[name] = y_pred

    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec  = recall_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    roc  = auc(fpr, tpr)

    cv_f1 = 0.0
    if cv_X is not None:
        skf = StratifiedKFold(n_splits=CONFIG['cv_folds'],
                              shuffle=True, random_state=SEED)
        cv_scores = cross_val_score(model, cv_X, cv_y,
                                    cv=skf, scoring='f1', n_jobs=-1)
        cv_f1 = cv_scores.mean()

    tiers = [get_confidence_tier(p)['tier'] for p in y_prob]
    uncertain_pct = tiers.count('UNCERTAIN') / len(tiers) * 100

    print(f'\n{"="*56}')
    print(f'  RESULTS: {name}')
    print(f'{"="*56}')
    print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
    print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
    print(f'  F1-Score  : {f1:.4f}  ({f1*100:.2f}%)')
    print(f'  AUC-ROC   : {roc:.4f}  ({roc*100:.2f}%)')
    if cv_f1 > 0:
        print(f'  CV-F1     : {cv_f1:.4f}  ({cv_f1*100:.2f}%)')
    print(f'  Inference : {elapsed*1000:.1f}ms')
    print(f'  Uncertain : {uncertain_pct:.1f}% of predictions')
    print()
    print(classification_report(y_te, y_pred,
                                target_names=['Real','Fake']))

    # Confusion matrix
    cm = confusion_matrix(y_te, y_pred)
    fig, ax = plt.subplots(figsize=(7,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real','Fake'],
                yticklabels=['Real','Fake'],
                linewidths=0.5, annot_kws={'size':14,'weight':'bold'})
    ax.set_title(f'Confusion Matrix — {name}',
                 fontweight='bold', fontsize=13, pad=15)
    ax.set_ylabel('Actual Label')
    ax.set_xlabel('Predicted Label')
    tn, fp, fn, tp = cm.ravel()
    ax.text(0.5,-0.14,
            f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}',
            ha='center', transform=ax.transAxes, fontsize=10)
    plt.tight_layout()
    fname = OUTPUT+f'cm_{name.lower().replace(" ","_")}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()

    return {
        'Model'        : name,
        'Accuracy'     : round(acc,  4),
        'Precision'    : round(prec, 4),
        'Recall'       : round(rec,  4),
        'F1-Score'     : round(f1,   4),
        'AUC-ROC'      : round(roc,  4),
        'CV-F1'        : round(cv_f1,4),
        'Inference_ms' : round(elapsed*1000,1),
        'Uncertain_pct': round(uncertain_pct,1),
        'Category'     : 'Classical' if name not in
                         ['BiLSTM','RoBERTa'] else name
    }

print('Confidence framework ready')

---
## Section 8: Classical Models

In [ ]:
print('[STEP 6] Training Classical Models...')

# Naive Bayes
print('\n[6.1] Naive Bayes')
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)
nb_r = full_evaluation('Naive Bayes', nb,
    X_train_tfidf, y_train, X_test_tfidf, y_test,
    cv_X=X_train_tfidf, cv_y=y_train)
all_results.append(nb_r)
print('Naive Bayes done')

# Logistic Regression
print('\n[6.2] Logistic Regression')
lr = LogisticRegression(C=1.0, max_iter=1000,
                        solver='lbfgs', random_state=SEED, n_jobs=-1)
lr.fit(X_train_tfidf, y_train)
lr_r = full_evaluation('Logistic Regression', lr,
    X_train_tfidf, y_train, X_test_tfidf, y_test,
    cv_X=X_train_tfidf, cv_y=y_train)
all_results.append(lr_r)
print('Logistic Regression done')

# SVM
print('\n[6.3] Support Vector Machine')
svm_base = LinearSVC(C=1.0, max_iter=3000, random_state=SEED)
svm = CalibratedClassifierCV(svm_base, cv=3)
svm.fit(X_train_tfidf, y_train)
svm_r = full_evaluation('Support Vector Machine', svm,
    X_train_tfidf, y_train, X_test_tfidf, y_test,
    cv_X=X_train_tfidf, cv_y=y_train)
all_results.append(svm_r)
print('SVM done')

# Random Forest
print('\n[6.4] Random Forest')
rf = RandomForestClassifier(n_estimators=200, max_features='sqrt',
                            random_state=SEED, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_r = full_evaluation('Random Forest', rf,
    X_train_tfidf, y_train, X_test_tfidf, y_test,
    cv_X=X_train_tfidf, cv_y=y_train)
all_results.append(rf_r)
print('Random Forest done')

print('\nALL CLASSICAL MODELS COMPLETE')

---
## Section 9: BiLSTM Model

In [ ]:
print('[STEP 7] Training BiLSTM Model...')

tok_lstm = Tokenizer(num_words=CONFIG['vocab_size'], oov_token='<OOV>')
tok_lstm.fit_on_texts(X_train)

def to_pad(texts):
    return pad_sequences(
        tok_lstm.texts_to_sequences(texts),
        maxlen=CONFIG['max_sequence_length'],
        padding='post', truncating='post')

X_tr_pad  = to_pad(X_train)
X_val_pad = to_pad(X_val)
X_te_pad  = to_pad(X_test)

inp = Input(shape=(CONFIG['max_sequence_length'],))
x   = Embedding(CONFIG['vocab_size'], CONFIG['embedding_dim'],
                input_length=CONFIG['max_sequence_length'])(inp)
x   = Dropout(0.2)(x)
x   = Bidirectional(LSTM(CONFIG['lstm_units'], return_sequences=True,
                         dropout=0.3, recurrent_dropout=0.2))(x)
x   = Bidirectional(LSTM(CONFIG['lstm_units']//2, return_sequences=True,
                         dropout=0.3, recurrent_dropout=0.2))(x)
x   = Conv1D(128, 3, activation='relu', padding='same')(x)
x   = GlobalMaxPooling1D()(x)
x   = Dense(256, activation='relu')(x)
x   = Dropout(CONFIG['lstm_dropout'])(x)
x   = Dense(128, activation='relu')(x)
x   = Dropout(0.2)(x)
out = Dense(1, activation='sigmoid')(x)
lstm_model = Model(inp, out)

lstm_model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')])

callbacks = [
    EarlyStopping(monitor='val_auc', patience=3,
                  restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=2, min_lr=1e-6),
    ModelCheckpoint(OUTPUT+'best_lstm.keras',
                    monitor='val_auc', save_best_only=True,
                    mode='max', verbose=0)
]

history = lstm_model.fit(
    X_tr_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=CONFIG['lstm_epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks, verbose=1)

# Training history chart
fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('BiLSTM Training History', fontsize=14, fontweight='bold')
for i, (m, t) in enumerate(zip(['loss','accuracy','auc'],
                                ['Loss','Accuracy','AUC'])):
    axes[i].plot(history.history[m], label='Train',
                 color=COLORS['real'], linewidth=2)
    axes[i].plot(history.history[f'val_{m}'], label='Validation',
                 color=COLORS['fake'], linewidth=2)
    axes[i].set_title(t, fontweight='bold')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT+'03_lstm_training.png', dpi=150, bbox_inches='tight')
plt.close()

# Evaluate
lstm_probs = lstm_model.predict(X_te_pad, verbose=0).flatten()
lstm_preds = (lstm_probs >= 0.5).astype(int)
all_probs['BiLSTM'] = lstm_probs
all_preds['BiLSTM'] = lstm_preds

acc_l  = accuracy_score(y_test, lstm_preds)
prec_l = precision_score(y_test, lstm_preds)
rec_l  = recall_score(y_test, lstm_preds)
f1_l   = f1_score(y_test, lstm_preds)
fpr_l, tpr_l, _ = roc_curve(y_test, lstm_probs)
auc_l  = auc(fpr_l, tpr_l)
uncertain_l = sum(1 for p in lstm_probs
                  if get_confidence_tier(p)['tier']=='UNCERTAIN')
uncertain_l_pct = uncertain_l/len(lstm_probs)*100

print(f'\nBiLSTM Results:')
print(f'  Accuracy  : {acc_l:.4f}')
print(f'  F1-Score  : {f1_l:.4f}')
print(f'  AUC-ROC   : {auc_l:.4f}')
print(classification_report(y_test, lstm_preds,
                            target_names=['Real','Fake']))

cm_l = confusion_matrix(y_test, lstm_preds)
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(cm_l, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real','Fake'], yticklabels=['Real','Fake'],
            linewidths=0.5, annot_kws={'size':14,'weight':'bold'})
ax.set_title('Confusion Matrix — BiLSTM', fontweight='bold', fontsize=13)
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(OUTPUT+'cm_bilstm.png', dpi=150, bbox_inches='tight')
plt.close()

all_results.append({
    'Model':'BiLSTM','Accuracy':round(acc_l,4),
    'Precision':round(prec_l,4),'Recall':round(rec_l,4),
    'F1-Score':round(f1_l,4),'AUC-ROC':round(auc_l,4),
    'CV-F1':0.0,'Inference_ms':0.0,
    'Category':'Deep Learning',
    'Uncertain_pct':round(uncertain_l_pct,1)
})

# Save LSTM
lstm_model.save(OUTPUT+'model_lstm.keras')
pickle.dump(tok_lstm, open(OUTPUT+'lstm_tokenizer.pkl','wb'))
print('BiLSTM complete and saved')

---
## Section 10: RoBERTa Fine-Tuning

In [ ]:
OUTPUT = '/kaggle/working/'

print('[STEP 8] Fine-tuning RoBERTa...')
print(f'Model: {CONFIG["roberta_model"]}')
print('Estimated time: 15-30 minutes on GPU\n')

MAX_TRAIN = 20000
MAX_TEST  = 5000

def stratified_sample(X, y, n):
    idx = []
    for cls in [0, 1]:
        cls_idx = y[y==cls].index.tolist()
        sampled = np.random.choice(
            cls_idx, min(len(cls_idx), n//2), replace=False)
        idx.extend(sampled)
    np.random.shuffle(idx)
    return X.loc[idx].tolist(), y.loc[idx].tolist()

X_tr_rb, y_tr_rb = stratified_sample(X_train, y_train, MAX_TRAIN)
X_te_rb, y_te_rb = stratified_sample(X_test,  y_test,  MAX_TEST)
print(f'Training: {len(X_tr_rb):,} | Test: {len(X_te_rb):,}')

rb_tok = AutoTokenizer.from_pretrained(CONFIG['roberta_model'])

def tokenize_fn(examples):
    return rb_tok(examples['text'], truncation=True,
                  padding=False, max_length=256)

tr_ds = Dataset.from_dict({'text': X_tr_rb, 'label': y_tr_rb})
te_ds = Dataset.from_dict({'text': X_te_rb, 'label': y_te_rb})
tr_ds = tr_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
te_ds = te_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

rb_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['roberta_model'], num_labels=2)

collator = DataCollatorWithPadding(tokenizer=rb_tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=-1)
    probs  = tf.nn.softmax(logits, axis=-1).numpy()[:, 1]
    fpr_r, tpr_r, _ = roc_curve(labels, probs)
    return {
        'accuracy' : accuracy_score(labels, preds),
        'f1'       : f1_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall'   : recall_score(labels, preds, zero_division=0),
        'auc'      : auc(fpr_r, tpr_r)
    }

total_steps  = (len(X_tr_rb) // CONFIG['roberta_batch_size']) * CONFIG['roberta_epochs']
warmup_steps = int(total_steps * 0.1)

args = TrainingArguments(
    output_dir=OUTPUT+'rb_output',
    num_train_epochs=CONFIG['roberta_epochs'],
    per_device_train_batch_size=CONFIG['roberta_batch_size'],
    per_device_eval_batch_size=CONFIG['roberta_batch_size'],
    learning_rate=CONFIG['roberta_lr'],
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED)

trainer = Trainer(
    model=rb_model,
    args=args,
    train_dataset=tr_ds,
    eval_dataset=te_ds,
    data_collator=collator,
    compute_metrics=compute_metrics)

trainer.train()
rb_eval = trainer.evaluate()

rb_out       = trainer.predict(te_ds)
rb_preds_arr = np.argmax(rb_out.predictions, axis=-1)
rb_probs_arr = tf.nn.softmax(
    rb_out.predictions, axis=-1).numpy()[:, 1]
all_probs['RoBERTa'] = rb_probs_arr
all_preds['RoBERTa'] = rb_preds_arr

uncertain_rb     = sum(1 for p in rb_probs_arr
    if get_confidence_tier(p)['tier'] == 'UNCERTAIN')
uncertain_rb_pct = uncertain_rb / len(rb_probs_arr) * 100

print(f'\nRoBERTa Results:')
print(f'  Accuracy  : {rb_eval["eval_accuracy"]:.4f}')
print(f'  F1-Score  : {rb_eval["eval_f1"]:.4f}')
print(f'  AUC-ROC   : {rb_eval["eval_auc"]:.4f}')

cm_rb = confusion_matrix(y_te_rb, rb_preds_arr)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_rb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real','Fake'],
            yticklabels=['Real','Fake'],
            linewidths=0.5,
            annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Confusion Matrix - RoBERTa',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(OUTPUT+'cm_roberta.png', dpi=150, bbox_inches='tight')
plt.close()

all_results.append({
    'Model'        : 'RoBERTa',
    'Accuracy'     : round(rb_eval['eval_accuracy'],  4),
    'Precision'    : round(rb_eval['eval_precision'], 4),
    'Recall'       : round(rb_eval['eval_recall'],    4),
    'F1-Score'     : round(rb_eval['eval_f1'],        4),
    'AUC-ROC'      : round(rb_eval['eval_auc'],       4),
    'CV-F1'        : 0.0,
    'Inference_ms' : 0.0,
    'Category'     : 'Transformer',
    'Uncertain_pct': round(uncertain_rb_pct, 1)
})

trainer.save_model(OUTPUT+'model_roberta')
rb_tok.save_pretrained(OUTPUT+'model_roberta')
print('RoBERTa complete and saved')


---
## Section 11: Full Comparative Analysis

In [ ]:
print('[STEP 9] Full Comparative Analysis...')

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(
    'F1-Score', ascending=False).reset_index(drop=True)
results_df.index += 1

print('\n' + '='*70)
print('  COMPLETE MODEL COMPARISON')
print('='*70)
print(results_df[['Model','Category','Accuracy',
                   'Precision','Recall','F1-Score',
                   'AUC-ROC','Uncertain_pct']].to_string())

best_model = results_df.iloc[0]['Model']
best_f1    = results_df.iloc[0]['F1-Score']
print(f'\nBest Model : {best_model}')
print(f'Best F1    : {best_f1:.4f} ({best_f1*100:.2f}%)')

models = results_df['Model'].tolist()
colors = ['#1F3864','#2E75B6','#70AD47',
          '#FFC000','#C00000','#7030A0']

fig = plt.figure(figsize=(20,12))
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        hspace=0.4, wspace=0.35)

# F1 bar
ax1 = fig.add_subplot(gs[0,0])
bars = ax1.barh(models, results_df['F1-Score'],
                color=colors, alpha=0.88)
ax1.set_xlim(0.7,1.01)
ax1.set_title('F1-Score', fontweight='bold')
for bar, val in zip(bars, results_df['F1-Score']):
    ax1.text(val+0.002, bar.get_y()+bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

# AUC bar
ax2 = fig.add_subplot(gs[0,1])
bars2 = ax2.barh(models, results_df['AUC-ROC'],
                 color=colors, alpha=0.88)
ax2.set_xlim(0.7,1.01)
ax2.set_title('AUC-ROC', fontweight='bold')
for bar, val in zip(bars2, results_df['AUC-ROC']):
    ax2.text(val+0.002, bar.get_y()+bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

# Uncertainty
ax3 = fig.add_subplot(gs[0,2])
bars3 = ax3.barh(models, results_df['Uncertain_pct'],
                 color=colors, alpha=0.88)
ax3.set_title('Uncertain Predictions (%)', fontweight='bold')
for bar, val in zip(bars3, results_df['Uncertain_pct']):
    ax3.text(val+0.1, bar.get_y()+bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=9)

# All metrics grouped
ax4 = fig.add_subplot(gs[1,0:2])
metrics = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC']
x = np.arange(len(models))
w = 0.15
mc = ['#1F3864','#2E75B6','#70AD47','#C00000','#7030A0']
for i,(m,c) in enumerate(zip(metrics,mc)):
    ax4.bar(x+i*w, results_df[m], w, label=m, color=c, alpha=0.85)
ax4.set_xticks(x+w*2)
ax4.set_xticklabels(models, rotation=30, ha='right', fontsize=9)
ax4.set_ylim(0.7,1.05)
ax4.set_title('All Metrics Comparison', fontweight='bold')
ax4.legend(fontsize=8)

# Radar
ax5 = fig.add_subplot(gs[1,2], projection='polar')
radar_m = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC']
n_vars  = len(radar_m)
angles  = np.linspace(0,2*np.pi,n_vars,endpoint=False).tolist()
angles += angles[:1]
for i,(_,row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in radar_m]+[row[radar_m[0]]]
    ax5.plot(angles,vals,'o-',linewidth=2,
             color=colors[i],label=row['Model'],alpha=0.85)
    ax5.fill(angles,vals,alpha=0.07,color=colors[i])
ax5.set_xticks(angles[:-1])
ax5.set_xticklabels(radar_m, fontsize=9)
ax5.set_ylim(0.7,1.0)
ax5.set_title('Radar Chart', fontweight='bold', pad=20)
ax5.legend(loc='upper right',
           bbox_to_anchor=(1.4,1.15), fontsize=7)

fig.suptitle(
    'Fake News Detection — Complete Model Comparison\n'
    'Arafat Hazrati | 24015414 | London Metropolitan University',
    fontsize=13, fontweight='bold')
plt.savefig(OUTPUT+'04_complete_comparison.png',
            dpi=150, bbox_inches='tight')
plt.close()

# ROC curves
fig, ax = plt.subplots(figsize=(10,8))
roc_classical = {
    'Naive Bayes':(nb,X_test_tfidf,y_test),
    'Logistic Regression':(lr,X_test_tfidf,y_test),
    'Support Vector Machine':(svm,X_test_tfidf,y_test),
    'Random Forest':(rf,X_test_tfidf,y_test),
}
for (name,(model,X_te,y_te)),color in zip(
        roc_classical.items(), colors):
    probs = model.predict_proba(X_te)[:,1]
    fpr,tpr,_ = roc_curve(y_te,probs)
    roc_auc = auc(fpr,tpr)
    ax.plot(fpr,tpr,color=color,linewidth=2.5,
            label=f'{name} (AUC={roc_auc:.4f})')
fpr_l,tpr_l,_ = roc_curve(y_test,lstm_probs)
ax.plot(fpr_l,tpr_l,color='#C00000',linewidth=2.5,
        label=f'BiLSTM (AUC={auc(fpr_l,tpr_l):.4f})')
fpr_r,tpr_r,_ = roc_curve(y_te_rb,rb_probs_arr)
ax.plot(fpr_r,tpr_r,color='#7030A0',linewidth=2.5,
        label=f'RoBERTa (AUC={auc(fpr_r,tpr_r):.4f})')
ax.plot([0,1],[0,1],'k--',linewidth=1.5,alpha=0.5)
ax.set_xlabel('False Positive Rate',fontsize=12)
ax.set_ylabel('True Positive Rate',fontsize=12)
ax.set_title('ROC Curves — All Models',fontweight='bold',fontsize=14)
ax.legend(loc='lower right',fontsize=9)
ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT+'05_roc_curves.png',dpi=150,bbox_inches='tight')
plt.close()
print('Comparison charts saved')

---
## Section 12: Error Analysis

In [ ]:
print('[STEP 10] Error Analysis...')

best_classical = results_df[
    results_df['Category']=='Classical'].iloc[0]['Model']
best_clf_probs = all_probs[best_classical]
best_clf_preds = all_preds[best_classical]

test_articles = X_test.reset_index(drop=True)
test_labels   = y_test.reset_index(drop=True)

error_df = pd.DataFrame({
    'article'    : test_articles,
    'true_label' : test_labels,
    'pred_label' : best_clf_preds,
    'confidence' : best_clf_probs
})

fp_df = error_df[
    (error_df['true_label']==0) &
    (error_df['pred_label']==1)
].sort_values('confidence', ascending=False)

fn_df = error_df[
    (error_df['true_label']==1) &
    (error_df['pred_label']==0)
].sort_values('confidence', ascending=True)

N = CONFIG['error_analysis_n']
print(f'False Positives (Real flagged as Fake): {len(fp_df):,}')
print(f'False Negatives (Fake missed)         : {len(fn_df):,}')

fig, axes = plt.subplots(1,2,figsize=(16,6))
fig.suptitle(f'Error Analysis — {best_classical}',
             fontweight='bold',fontsize=13)
axes[0].hist(fp_df['confidence'],bins=30,
             color=COLORS['fake'],alpha=0.8,edgecolor='white')
axes[0].set_title('False Positives',fontweight='bold')
axes[0].set_xlabel('Confidence Score (Fake)')
axes[1].hist(fn_df['confidence'],bins=30,
             color=COLORS['real'],alpha=0.8,edgecolor='white')
axes[1].set_title('False Negatives',fontweight='bold')
axes[1].set_xlabel('Confidence Score (Real — incorrectly)')
plt.tight_layout()
plt.savefig(OUTPUT+'07_error_analysis.png',dpi=150,bbox_inches='tight')
plt.close()

fp_df.head(N).to_csv(OUTPUT+'false_positives.csv',index=False)
fn_df.head(N).to_csv(OUTPUT+'false_negatives.csv',index=False)
print('Error analysis complete')

---
## Section 13: Real-World Evaluation

In [ ]:
print('[STEP 11] Real-World Evaluation...')

real_world_examples = [
    {'text': 'The Federal Reserve raised interest rates by 25 basis points '
             'on Wednesday as the central bank continues its effort to bring '
             'inflation back to its 2 percent target.',
     'true_label': 0, 'source': 'Reuters-style financial news'},
    {'text': 'BREAKING: Scientists discover that drinking bleach cures all '
             'diseases. Mainstream media suppressing this. Big Pharma does '
             'not want you to know. Share before they delete it.',
     'true_label': 1, 'source': 'Health misinformation'},
    {'text': 'The United Nations General Assembly voted to adopt a resolution '
             'calling for an immediate ceasefire with 143 member states voting '
             'in favour and 7 against.',
     'true_label': 0, 'source': 'UN-style reporting'},
    {'text': 'SHOCKING: The president was secretly replaced by a body double. '
             'Insider sources confirm the real leader is held underground. '
             'Wake up people, the truth is being hidden.',
     'true_label': 1, 'source': 'Conspiracy theory'},
    {'text': 'Apple reported quarterly earnings of 94.8 billion dollars '
             'beating analyst expectations as iPhone sales remained strong '
             'despite macroeconomic headwinds.',
     'true_label': 0, 'source': 'Corporate earnings'},
    {'text': 'New study proves 5G towers transmit mind-control signals causing '
             'mass compliance. Government officials refuse to comment. '
             'Protect yourself now.',
     'true_label': 1, 'source': 'Technology conspiracy'},
    {'text': 'The European Central Bank kept its key interest rates unchanged '
             'citing easing inflationary pressures and weaker economic growth '
             'in the eurozone during the third quarter.',
     'true_label': 0, 'source': 'Central bank news'},
    {'text': 'EXCLUSIVE: Celebrity admits moon landing was faked and earth is '
             'flat. Hollywood insiders confirm all major stars are in on it. '
             'You will not believe what they are hiding.',
     'true_label': 1, 'source': 'Celebrity conspiracy'},
    {'text': 'A magnitude 6.4 earthquake struck the coastal region on Friday '
             'morning according to the national geological survey. No casualties '
             'reported and authorities are monitoring for aftershocks.',
     'true_label': 0, 'source': 'Natural disaster'},
    {'text': 'Government microchips secretly inserted into vaccines to track '
             'population. Whistleblower from pharmaceutical company released '
             'documents proving this since 2019. Share everywhere.',
     'true_label': 1, 'source': 'Vaccine misinformation'},
]

rw_df = pd.DataFrame(real_world_examples)
rw_df['clean'] = rw_df['text'].apply(clean_text)
rw_tfidf = tfidf.transform(rw_df['clean'])
rw_probs = lr.predict_proba(rw_tfidf)[:,1]
rw_preds = (rw_probs >= 0.5).astype(int)

rw_df['predicted']  = rw_preds
rw_df['confidence'] = rw_probs
rw_df['correct']    = (rw_df['predicted']==rw_df['true_label'])
rw_df['verdict']    = rw_df['confidence'].apply(
    lambda p: get_confidence_tier(p)['label'])

rw_accuracy = rw_df['correct'].mean()
lab_accuracy = results_df[
    results_df['Model']==best_classical]['Accuracy'].values[0]

print(f'Real-World Accuracy : {rw_accuracy*100:.1f}%')
print(f'Lab Accuracy (ISOT) : {lab_accuracy*100:.2f}%')
print(f'Performance Gap     : {(lab_accuracy-rw_accuracy)*100:.1f} pp')
print()
for _,row in rw_df.iterrows():
    true_str = 'REAL' if row['true_label']==0 else 'FAKE'
    result   = 'CORRECT' if row['correct'] else 'WRONG'
    print(f'  {result} | True: {true_str} | Predicted: {row["verdict"]} | {row["source"]}')

rw_df.to_csv(OUTPUT+'real_world_evaluation.csv',index=False)
print('Real-world evaluation complete')

---
## Section 14: LIME Explainability

In [ ]:
print('[STEP 12] LIME Explainability...')

explainer = LimeTextExplainer(
    class_names=['Real News','Fake News'],
    random_state=SEED)

def predict_proba_pipeline(texts):
    cleaned     = [clean_text(t) for t in texts]
    transformed = tfidf.transform(cleaned)
    return lr.predict_proba(transformed)

test_articles_list = X_test.reset_index(drop=True)
test_labels_list   = y_test.reset_index(drop=True)
lr_probs_all = all_probs['Logistic Regression']

high_conf_real = np.where((test_labels_list==0) & (lr_probs_all<0.1))[0]
high_conf_fake = np.where((test_labels_list==1) & (lr_probs_all>0.9))[0]
uncertain_idx  = np.where(np.abs(lr_probs_all-0.5)<0.08)[0]
wrong_idx      = np.where(all_preds['Logistic Regression']!=test_labels_list.values)[0]

explain_cases = []
if len(high_conf_real)>0:
    explain_cases.append({'idx':high_conf_real[0],
        'case':'High Confidence Real','true':0})
if len(high_conf_fake)>0:
    explain_cases.append({'idx':high_conf_fake[0],
        'case':'High Confidence Fake','true':1})
if len(uncertain_idx)>0:
    explain_cases.append({'idx':uncertain_idx[0],
        'case':'Uncertain Prediction',
        'true':int(test_labels_list[uncertain_idx[0]])})
if len(wrong_idx)>0:
    explain_cases.append({'idx':wrong_idx[0],
        'case':'Incorrect Prediction',
        'true':int(test_labels_list[wrong_idx[0]])})

lime_explanations = []

for case in explain_cases:
    idx  = case['idx']
    text = test_articles_list[idx]
    prob = lr_probs_all[idx]
    tier = get_confidence_tier(prob)

    print(f'\nExplaining: {case["case"]}')
    print(f'  True    : {"Real" if case["true"]==0 else "Fake"}')
    print(f'  Verdict : {tier["label"]} ({prob*100:.1f}% fake)')

    exp = explainer.explain_instance(
        text, predict_proba_pipeline,
        num_features=CONFIG['lime_num_features'],
        num_samples=CONFIG['lime_num_samples'],
        labels=[0,1])

    feature_weights = exp.as_list(label=1)
    pos_words = [(w,v) for w,v in feature_weights if v>0]
    neg_words = [(w,v) for w,v in feature_weights if v<0]

    print('  Top words pushing FAKE:')
    for word,weight in sorted(pos_words,
                              key=lambda x:x[1],reverse=True)[:5]:
        print(f'    + {word:<20} ({weight:+.4f})')
    print('  Top words pushing REAL:')
    for word,weight in sorted(neg_words,key=lambda x:x[1])[:5]:
        print(f'    - {word:<20} ({weight:+.4f})')

    lime_explanations.append({
        'case'          :case['case'],
        'true_label'    :case['true'],
        'confidence_fake':prob,
        'tier'          :tier['tier'],
        'top_fake_words':[w for w,v in pos_words[:5]],
        'top_real_words':[w for w,v in neg_words[:5]]
    })

    fig = exp.as_pyplot_figure(label=1)
    plt.title(f'{case["case"]} — {tier["label"]} ({prob*100:.1f}%)',
              fontweight='bold')
    plt.tight_layout()
    fname = OUTPUT+f'lime_{case["case"].lower().replace(" ","_")[:25]}.png'
    plt.savefig(fname,dpi=150,bbox_inches='tight')
    plt.close()

with open(OUTPUT+'lime_explanations.json','w') as f:
    json.dump(lime_explanations, f, indent=2)
print('LIME explainability complete')

---
## Section 15: Save All Models

In [ ]:
print('[STEP 13] Saving All Models...')

pickle.dump(nb,    open(OUTPUT+'model_naive_bayes.pkl',         'wb'))
pickle.dump(lr,    open(OUTPUT+'model_logistic_regression.pkl', 'wb'))
pickle.dump(svm,   open(OUTPUT+'model_svm.pkl',                 'wb'))
pickle.dump(rf,    open(OUTPUT+'model_random_forest.pkl',       'wb'))
pickle.dump(tfidf, open(OUTPUT+'tfidf_vectorizer.pkl',          'wb'))

# Save config for API
api_config = {
    'best_model'           : best_model,
    'confidence_high'      : CONFIG['confidence_high'],
    'confidence_medium'    : CONFIG['confidence_medium'],
    'confidence_uncertain' : CONFIG['confidence_uncertain'],
    'max_sequence_length'  : CONFIG['max_sequence_length'],
    'vocab_size'           : CONFIG['vocab_size'],
}
with open(OUTPUT+'api_config.json','w') as f:
    json.dump(api_config, f, indent=2)

results_df.to_csv(OUTPUT+'complete_results.csv', index=False)

print('All models saved')
print()

# Final summary
print('='*65)
print('  FINAL RESULTS SUMMARY')
print('  Fake News Detection — Arafat Hazrati | 24015414')
print('='*65)
medals = ['1st','2nd','3rd','4th','5th','6th']
for i,row in results_df.iterrows():
    print(f'  {medals[i-1]}  {row["Model"]:<26} '
          f'F1: {row["F1-Score"]:.4f}  '
          f'AUC: {row["AUC-ROC"]:.4f}')
print('='*65)
print(f'  Best Model : {best_model}')
print(f'  Best F1    : {best_f1:.4f} ({best_f1*100:.2f}%)')
print('='*65)
print()
print('All files saved to /kaggle/working/')
print('Go to Output tab on the right to download everything.')
print('PIPELINE COMPLETE')